In [1]:
# Install necessary libraries (Uncomment if required)
# !pip install pdfplumber llama-index

# Import necessary libraries
import json
from llama_index.llms.groq import Groq
from llama_index.core.llms import ChatMessage


In [2]:
llm = Groq(model="llama3-70b-8192", api_key="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")


In [3]:
import pdfplumber
import asyncio

# Asynchronous function to extract text from PDF
async def extract_text_from_pdf(pdf_path):
    all_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            print(f"Total pages in PDF: {total_pages}")
            
            for i, page in enumerate(pdf.pages):
                await asyncio.sleep(0.1)  # Simulate some delay for processing
                page_text = page.extract_text()
                if page_text:
                    all_text += page_text  # Add text of each page
                    print(f"Processed page {i+1}/{total_pages}")
                else:
                    print(f"Page {i+1} contains no text")
                    
        print("PDF processing complete.")
    except Exception as e:
        print(f"Error reading PDF file: {e}")
        return None
    
    return all_text


In [4]:
# Function to convert extracted text to JSON-like structure
def convert_text_to_json(text, chunk_size=5000):
    # Split the text into smaller chunks (e.g., per 5000 characters or page-wise)
    chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]
    
    # Create a JSON structure with each chunk labeled
    data = {"pages": [{"page": i + 1, "content": chunk} for i, chunk in enumerate(chunks)]}
    
    return json.dumps(data, indent=2)


In [5]:
# Function to search the JSON for relevant content based on keywords
def search_json_for_keywords(json_data, keywords):
    found_sections = []
    for page in json_data["pages"]:
        if any(keyword.lower() in page["content"].lower() for keyword in keywords):
            found_sections.append(page)
    return found_sections


In [6]:
# Function to construct the prompt from the JSON content
def construct_prompt_from_json(json_content, user_question):
    prompt = f"""
    You are an expert assistant tasked with providing information based only on the JSON content. 
    The user has asked a question, and below is the relevant content extracted from the document.

    JSON Content: {json_content}

    User Question: {user_question}

    Please provide an accurate and concise response based on the content provided from the JSON document.
    """
    return prompt


In [7]:
import asyncio

# Asynchronous function to handle chat interaction
async def get_chat_with_pdf(pdf_path):
    # Step 1: Asynchronously extract the text from the PDF file
    pdf_text = await extract_text_from_pdf(pdf_path)
    if not pdf_text:
        print("Unable to extract text from the PDF file.")
        return

    # Step 2: Convert the text to JSON-like format (structured by pages or chunks)
    json_content = convert_text_to_json(pdf_text)
    
    # Load JSON data
    json_data = json.loads(json_content)

    total_chunks = len(json_data["pages"])
    print(f"Total chunks to process: {total_chunks}")

    processed_chunks = 0  # Track how many chunks have been processed

    while True:
        user_question = input("You: ")
        if user_question.lower() in ["exit", "bye", "quit"]:
            print("bot: Goodbye!")
            break

        # Step 3: Search the JSON content for relevant sections based on user query
        question_keywords = user_question.split()  # Split user question into keywords
        relevant_sections = search_json_for_keywords(json_data, question_keywords)

        # If relevant sections are found, process each and send to the chatbot
        if relevant_sections:
            for section in relevant_sections:
                processed_chunks += 1
                prompt = construct_prompt_from_json(section["content"], user_question)
                messages = [ChatMessage(role="system", content=prompt)]

                # Get the response from the chatbot
                response = llm.chat(messages)

                # Print the bot's response
                final_response = response.message.content
                print(f"bot: {final_response}")
        else:
            print("bot: No relevant content found in the document.")
    
    print(f"Processed {processed_chunks} out of {total_chunks} chunks.")


In [ ]:
pdf_file_path = './decrypted_output.pdf'
await get_chat_with_pdf(pdf_file_path)


Total pages in PDF: 128
Processed page 1/128
Processed page 2/128
Processed page 3/128
Page 4 contains no text
Processed page 5/128
Processed page 6/128
Processed page 7/128
Processed page 8/128
Processed page 9/128
Page 10 contains no text
Processed page 11/128
Page 12 contains no text
Processed page 13/128
Processed page 14/128
Processed page 15/128
Processed page 16/128
Processed page 17/128
Processed page 18/128
Processed page 19/128
Page 20 contains no text
Processed page 21/128
Processed page 22/128
Processed page 23/128
Processed page 24/128
Processed page 25/128
Processed page 26/128
Processed page 27/128
Processed page 28/128
Processed page 29/128
Page 30 contains no text
Processed page 31/128
Processed page 32/128
Processed page 33/128
Processed page 34/128
Processed page 35/128
Page 36 contains no text
Processed page 37/128
Processed page 38/128
Processed page 39/128
Processed page 40/128
Processed page 41/128
Page 42 contains no text
Processed page 43/128
Processed page 44/